# Structured extraction validation — Gemma 4 E4B-it

Validates whether the chosen VLM (`google/gemma-4-E4B-it`) can extract
structured outputs from real UFV campus images, using five extraction
prompts that cover the visual scenarios the indoor-navigation system
deals with.

Two output families, picked according to what each photo can carry:

- **PDDL location atoms** — `room_plate` (door plates), `directory_sign`
  (floor directories). The photo contains a unique campus-wide identifier
  (room number, floor letter), so the model returns a planner fact
  `(at user <location_id>)` consumed directly by Fast Downward as initial
  state.
- **JSON state objects** — `elevator_state`, `stair_direction`,
  `restroom_gender`. The photo carries only state or attribute information
  (door open/closed, stairs up/down, restroom male/female), not location.
  The model returns `{"type": ..., "detail": ...}` consumed by the
  conversation FSM as a runtime verification signal against the active
  plan.

Single-model validation, not a comparison. The question this notebook
answers is "does Gemma 4 E4B-it extract the expected outputs from real
campus photos?" — pass / fail per prompt.

## How to use

1. **Runtime → A100** (E4B in bfloat16 needs ~16 GB VRAM; T4 free won't fit).
2. Drag these files into `/content/` (Colab file panel):
   - `images.zip` (auto-extracts)
   - `prompts.json`
   - `ground_truth.json`
3. Paste your `HF_TOKEN` in §2.
4. Run §1 → §7 in order. The final cell zips `/content/results/` and triggers
   a browser download — you don't need to fish anything out of the runtime.


## 1. Install dependencies

If this is a fresh runtime, run this cell once and then `Runtime → Restart
runtime` (Pillow gets force-reinstalled, the kernel needs to reload it). On
restart you can skip §1 and start at §2.


In [ ]:
!pip install -qU --force-reinstall Pillow
!pip install -qU transformers accelerate pandas scikit-learn matplotlib seaborn
print()
print('=' * 60)
print('FIRST RUN ON FRESH RUNTIME: Runtime -> Restart runtime now,')
print('then run cells from sec 2 onwards (do NOT re-run sec 1).')
print('=' * 60)


## 2. Load Gemma 4 E4B-it

Loads the model once. Uses `dtype='auto'` (the canonical pattern from the
Gemma 4 model card) which resolves to bfloat16 on A100 and float32 on CPU.
Stays in VRAM until the runtime restarts, so §3-§7 can be re-run cheaply.

A fixed seed is set so the run is bit-for-bit reproducible (greedy decoding
makes this redundant, but the seed lands in the reproducibility log anyway).

Gated model — set `HF_TOKEN` below with a token from
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) that
has accepted the [model terms](https://huggingface.co/google/gemma-4-E4B-it).


In [ ]:
import time
import random
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

# === EDIT THIS ===
HF_TOKEN = ''  # paste your HuggingFace read token here
# ==================

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = 'google/gemma-4-E4B-it'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading {MODEL_ID} on {DEVICE} (dtype=auto, seed={SEED})...')
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    dtype='auto' if DEVICE == 'cuda' else torch.float32,
    device_map='auto' if DEVICE == 'cuda' else None,
).eval()
MODEL_LOAD_TIME_S = round(time.time() - t0, 2)

n_params = sum(p.numel() for p in model.parameters())
N_PARAMS_BILLIONS = round(n_params / 1e9, 2)
MODEL_DTYPE = str(model.dtype)
print(f'Loaded in {MODEL_LOAD_TIME_S} s -- {N_PARAMS_BILLIONS}B params (dtype={MODEL_DTYPE})')
if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')


## 3. Locate inputs and prepare workspace

Auto-detects the three input files anywhere under `/content/` (you can drop
them at the root, in subfolders, or inside an `annotations.zip` — the cell
handles all three layouts). Then extracts `images.zip` if needed and creates
the `results/` directory.


In [ ]:
import json
import shutil
import zipfile
from pathlib import Path
from collections import Counter

BASE_DIR = Path('/content')
RESULTS_DIR = BASE_DIR / 'results'
IMAGES_ROOT = BASE_DIR / 'images'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def find_file(name: str) -> Path:
    """Recursively locate a file under BASE_DIR. Raises if missing."""
    hits = list(BASE_DIR.rglob(name))
    # Filter out anything under results/ or sample_data/ to avoid stale copies.
    hits = [p for p in hits if 'results' not in p.parts and 'sample_data' not in p.parts]
    if not hits:
        raise FileNotFoundError(f'{name!r} not found anywhere under {BASE_DIR}')
    return hits[0]


# 1) annotations.zip is optional — if present, extract it first.
ann_zip = list(BASE_DIR.glob('annotations.zip'))
if ann_zip:
    print(f'Extracting {ann_zip[0]}...')
    with zipfile.ZipFile(ann_zip[0]) as z:
        z.extractall(BASE_DIR)

# 2) Locate prompts.json + ground_truth.json wherever they are.
prompts_path = find_file('prompts.json')
ground_truth_path = find_file('ground_truth.json')
print(f'Loaded prompts from:      {prompts_path}')
print(f'Loaded ground truth from: {ground_truth_path}')

with prompts_path.open('r', encoding='utf-8') as f:
    PROMPTS_DATA = json.load(f)
with ground_truth_path.open('r', encoding='utf-8') as f:
    GROUND_TRUTH = json.load(f)

# 3) Extract images.zip if the unpacked folder is missing.
img_zip = list(BASE_DIR.glob('images.zip'))
if not IMAGES_ROOT.exists() and img_zip:
    print(f'Extracting {img_zip[0]}...')
    with zipfile.ZipFile(img_zip[0]) as z:
        z.extractall(BASE_DIR)
    # Some zips nest under data/ufv_campus/images/ — flatten if so.
    nested = BASE_DIR / 'data' / 'ufv_campus' / 'images'
    if nested.exists() and not IMAGES_ROOT.exists():
        shutil.move(str(nested), str(IMAGES_ROOT))

if not IMAGES_ROOT.exists():
    raise FileNotFoundError(
        f'No {IMAGES_ROOT} folder and no images.zip to extract. Upload one and re-run.'
    )

# 4) Build the task list: one (image, prompt) per ground-truth entry.
TASKS = []
for rel_path, gt in GROUND_TRUTH.items():
    category = gt['category']
    matching = [pid for pid, p in PROMPTS_DATA['prompts'].items()
                if p['applies_to_category'] == category]
    if not matching:
        print(f'WARN: no prompt covers category {category!r} (skipping {rel_path})')
        continue
    TASKS.append({
        'image_rel_path': rel_path,
        'image_full_path': IMAGES_ROOT / rel_path,
        'category': category,
        'prompt_id': matching[0],
        'ground_truth': gt,
    })

print(f'\nTotal tasks: {len(TASKS)}')
print(f'Per category: {dict(Counter(t["category"] for t in TASKS))}')
print(f'Per prompt:   {dict(Counter(t["prompt_id"] for t in TASKS))}')

missing = [t['image_rel_path'] for t in TASKS if not t['image_full_path'].exists()]
if missing:
    print(f'\nMISSING IMAGES ({len(missing)}):')
    for m in missing[:10]:
        print(f'  - {m}')
    raise FileNotFoundError(f'{len(missing)} images missing from {IMAGES_ROOT}')
print('\nAll images present on disk.')


## 4. Run inference

One forward pass per image with the prompt that matches the image's category.
Timing brackets each call with `torch.cuda.synchronize()` so the latency
reflects actual GPU-side elapsed time, not Python overhead. Greedy decoding
(`do_sample=False`, `num_beams=1`, `enable_thinking=False`) so outputs are
deterministic per (image, prompt).

System prompt nudges the model to respect the requested format. User content
is a list of `{image, text}` blocks (image first per Gemma 4's modality-order
best practice).

`MAX_NEW_TOKENS = 30` is enough for any well-formed answer (the longest valid
output is ~25 chars) and keeps the model from generating trailing junk
tokens past the closing brace / parenthesis.


In [ ]:
from PIL import Image
import pandas as pd

SYSTEM_PROMPT = (
    'You are a strict format extractor for indoor navigation. '
    'Read the question, follow its rules exactly, and output only what the '
    'question asks. Never invent identifiers. Never add explanations or extra '
    'words. Stop generating immediately after the requested output is complete.'
)
MAX_NEW_TOKENS = 30


def run_one(image_path: Path, prompt_text: str) -> tuple[str, float]:
    image = Image.open(image_path).convert('RGB')
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': image},
            {'type': 'text',  'text': prompt_text},
        ]},
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
        enable_thinking=False,
    ).to(model.device)
    inputs = inputs.to(dtype=model.dtype)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    latency_ms = (time.time() - t0) * 1000

    input_len = inputs['input_ids'].shape[-1]
    response = processor.decode(output[0][input_len:], skip_special_tokens=True).strip()
    return response, latency_ms


raw_records = []
total_t0 = time.time()
for i, task in enumerate(TASKS, 1):
    prompt_text = PROMPTS_DATA['prompts'][task['prompt_id']]['prompt_text']
    try:
        response, latency_ms = run_one(task['image_full_path'], prompt_text)
        ok = True
        err = ''
    except Exception as e:
        response, latency_ms = '', 0.0
        ok = False
        err = repr(e)
    raw_records.append({
        'image_rel_path': task['image_rel_path'],
        'category': task['category'],
        'prompt_id': task['prompt_id'],
        'response': response,
        'latency_ms': round(latency_ms, 1),
        'inference_ok': ok,
        'error': err,
    })
    if i % 10 == 0 or i == len(TASKS):
        elapsed = time.time() - total_t0
        eta = (elapsed / i) * (len(TASKS) - i)
        print(f'[{i:3d}/{len(TASKS)}] {task["prompt_id"]:15s} {task["image_rel_path"]:55s} {latency_ms:7.1f} ms | elapsed {elapsed:6.1f}s ETA {eta:6.1f}s')

raw_df = pd.DataFrame(raw_records)
raw_df.to_csv(RESULTS_DIR / 'per_image_responses.csv', index=False)

VRAM_PEAK_GB = round(torch.cuda.max_memory_allocated() / 1024**3, 2) if torch.cuda.is_available() else 0.0
TOTAL_INFERENCE_TIME_S = round(time.time() - total_t0, 2)
print(f'\nDone. {len(raw_df)} inferences in {TOTAL_INFERENCE_TIME_S}s. Peak VRAM: {VRAM_PEAK_GB} GB.')
print(f'Saved per-image responses to {RESULTS_DIR / "per_image_responses.csv"}')


## 5. Parse responses and compute metrics

Two parsers, one per output family, plus a strict variant of each:

- **PDDL atoms** (`room_plate`, `directory_sign`):
  - Lenient `parse_pddl` — `(at user X)` regex search anywhere in the response, lowercased id.
  - Strict `parse_pddl_strict` — response must equal `(at user X)` exactly (no surrounding text, no casing flexibility).
- **JSON state** (`elevator_state`, `stair_direction`, `restroom_gender`):
  - Lenient `parse_json_state` — strips code fences, extracts the first `{...}` block, then `json.loads` (so trailing tokens after the closing brace don't break parsing).
  - Strict `parse_json_state_strict` — `json.loads` on the raw response; must parse and have the right `type` field.

Two `format_validity` columns are reported:
- `format_validity` (lenient) — "can our app pipeline consume this output?"
- `format_validity_strict` (strict) — "is the raw model emission character-perfect?"

`accuracy` uses the lenient extraction (a malformed-but-recoverable output still counts as wrong if the recovered value is wrong). For the three binary JSON prompts we additionally report **macro precision / recall / F1** computed on the format-valid subset, using the lenient parser.

Malformed outputs count as **incorrect** for accuracy (no leniency on correctness — a lenient parse that returns the wrong value is still wrong).


In [ ]:
import re

PDDL_ATOM_RE        = re.compile(r'\(\s*at\s+user\s+([a-zA-Z_][a-zA-Z0-9_]*)\s*\)')
PDDL_ATOM_STRICT_RE = re.compile(r'^\(at user ([a-z_][a-z0-9_]*)\)$')
JSON_OBJECT_RE      = re.compile(r'\{[^{}]*\}')

# Which prompt produces a PDDL atom and which produces a JSON state object.
PDDL_PROMPTS = {'room_plate', 'directory_sign'}
JSON_TYPE_BY_PROMPT = {
    'elevator_state':  'elevator',
    'stair_direction': 'stairs',
    'restroom_gender': 'restroom',
}


def expected_atom_for_room_plate(gt: dict) -> str:
    """Map a room_plates ground-truth entry to its canonical PDDL atom.

    Building G uses two-digit room codes "X.Y" -> classroom_g_X_Y.
    Building CD uses single-digit room codes "1".."8" -> classroom_cd_N
    (the digit is the unique room number campus-wide; no floor encoding
    needed because there are only 8 CD classrooms in total).
    """
    code = gt['room_code'].strip()
    if code.lower() == 'ai_lab':
        return '(at user ai_lab)'
    if '.' in code:
        a, b = code.split('.')
        return f'(at user classroom_g_{int(a)}_{int(b)})'
    n = int(code)
    if 1 <= n <= 8:
        return f'(at user classroom_cd_{n})'
    raise ValueError(f'Unmappable room_code: {code!r}')


def expected_atom_for_directory_sign(gt: dict) -> str:
    full = gt['expected_full'].strip()
    mapping = {
        'G 0':     '(at user directory_sign_g_0)',
        'G 1':     '(at user directory_sign_g_1)',
        'G 2':     '(at user directory_sign_g_2)',
        'G Agora': '(at user directory_sign_g_agora)',
    }
    if full not in mapping:
        raise ValueError(f'Unmappable expected_full: {full!r}')
    return mapping[full]


def expected_state_for_json(prompt_id: str, gt: dict) -> str:
    return {
        'elevator_state':  gt.get('elevator_state'),
        'stair_direction': gt.get('stairs_direction'),
        'restroom_gender': gt.get('bathroom_subtype'),
    }[prompt_id]


def parse_pddl(response: str) -> tuple[bool, str | None]:
    """Lenient PDDL parser: regex search anywhere, lowercase the id.

    Tolerates leading/trailing prose and casing differences. Used for
    accuracy because a recoverable response is still a usable answer
    once the production pipeline trims and lowercases.
    """
    if not response:
        return False, None
    m = PDDL_ATOM_RE.search(response.strip())
    if not m:
        return False, None
    return True, m.group(1).lower()


def parse_pddl_strict(response: str) -> bool:
    """Strict PDDL: response must equal '(at user X)' exactly, lowercase id."""
    if not response:
        return False
    return bool(PDDL_ATOM_STRICT_RE.match(response.strip()))


def parse_json_state(response: str, expected_type: str) -> tuple[bool, str | None]:
    """Lenient JSON parser: strips code fences, extracts the FIRST {...}
    block via regex (so trailing junk tokens after the closing brace don't
    break json.loads), then parses.
    """
    if not response:
        return False, None
    s = response.strip()
    s = re.sub(r'^```(?:json)?', '', s).strip()
    s = re.sub(r'```$', '', s).strip()
    match = JSON_OBJECT_RE.search(s)
    if not match:
        return False, None
    try:
        obj = json.loads(match.group(0))
    except Exception:
        return False, None
    if not isinstance(obj, dict):
        return False, None
    if obj.get('type') != expected_type:
        return False, None
    detail = obj.get('detail')
    if not isinstance(detail, str):
        return False, None
    return True, detail.lower()


def parse_json_state_strict(response: str, expected_type: str) -> bool:
    """Strict JSON: json.loads on the raw response, no fence stripping, no
    block extraction. Must parse to a dict with the right `type` and a
    string `detail` field.
    """
    if not response:
        return False
    try:
        obj = json.loads(response.strip())
    except Exception:
        return False
    if not isinstance(obj, dict):
        return False
    if obj.get('type') != expected_type:
        return False
    return isinstance(obj.get('detail'), str)


scored_records = []
for r, t in zip(raw_records, TASKS):
    pid = r['prompt_id']
    response = r['response']
    gt = t['ground_truth']

    if pid in PDDL_PROMPTS:
        if pid == 'room_plate':
            expected_atom = expected_atom_for_room_plate(gt)
        else:
            expected_atom = expected_atom_for_directory_sign(gt)
        expected_id = PDDL_ATOM_RE.search(expected_atom).group(1)
        format_valid, extracted = parse_pddl(response)
        format_valid_strict = parse_pddl_strict(response)
        accuracy_ok = bool(format_valid and extracted == expected_id.lower())
        scored_records.append({
            **r,
            'expected': expected_id,
            'extracted': extracted or '',
            'format_valid': format_valid,
            'format_valid_strict': format_valid_strict,
            'accuracy_ok':  accuracy_ok,
        })
    else:
        expected_type = JSON_TYPE_BY_PROMPT[pid]
        expected_detail = expected_state_for_json(pid, gt)
        format_valid, extracted = parse_json_state(response, expected_type)
        format_valid_strict = parse_json_state_strict(response, expected_type)
        accuracy_ok = bool(format_valid and extracted == (expected_detail or '').lower())
        scored_records.append({
            **r,
            'expected': expected_detail or '',
            'extracted': extracted or '',
            'format_valid': format_valid,
            'format_valid_strict': format_valid_strict,
            'accuracy_ok':  accuracy_ok,
        })

scored_df = pd.DataFrame(scored_records)
scored_df.to_csv(RESULTS_DIR / 'per_image_scored.csv', index=False)
print(f'Saved per-image scored results to {RESULTS_DIR / "per_image_scored.csv"}')
print(f'\nFirst few rows:')
print(scored_df[['prompt_id', 'expected', 'extracted', 'format_valid', 'format_valid_strict', 'accuracy_ok']].head(8).to_string(index=False))


In [ ]:
# Per-prompt summary metrics.
summary_rows = []
for pid in PROMPTS_DATA['prompts']:
    sub = scored_df[scored_df['prompt_id'] == pid]
    if len(sub) == 0:
        continue
    latencies = sub.loc[sub['inference_ok'], 'latency_ms'].values
    mean_ms = float(np.mean(latencies)) if len(latencies) else 0.0
    summary_rows.append({
        'prompt_id': pid,
        'n_imgs': len(sub),
        'format_validity':        round(sub['format_valid'].mean(), 4),
        'format_validity_strict': round(sub['format_valid_strict'].mean(), 4),
        'accuracy':               round(sub['accuracy_ok'].mean(), 4),
        'mean_ms':                round(mean_ms, 1),
        'std_ms':                 round(float(np.std(latencies)), 1) if len(latencies) else 0.0,
        'p95_ms':                 round(float(np.percentile(latencies, 95)), 1) if len(latencies) else 0.0,
        'throughput_inf_per_s':   round(1000.0 / mean_ms, 3) if mean_ms else 0.0,
    })

# Overall row.
all_lat = scored_df.loc[scored_df['inference_ok'], 'latency_ms'].values
overall_mean = float(all_lat.mean()) if len(all_lat) else 0.0
overall = {
    'prompt_id': 'TOTAL',
    'n_imgs': len(scored_df),
    'format_validity':        round(scored_df['format_valid'].mean(), 4),
    'format_validity_strict': round(scored_df['format_valid_strict'].mean(), 4),
    'accuracy':               round(scored_df['accuracy_ok'].mean(), 4),
    'mean_ms':                round(overall_mean, 1),
    'std_ms':                 round(float(all_lat.std()), 1) if len(all_lat) else 0.0,
    'p95_ms':                 round(float(np.percentile(all_lat, 95)), 1) if len(all_lat) else 0.0,
    'throughput_inf_per_s':   round(1000.0 / overall_mean, 3) if overall_mean else 0.0,
}
summary_rows.append(overall)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_DIR / 'per_prompt_summary.csv', index=False)
print('Per-prompt summary:')
print(summary_df.to_string(index=False))


In [ ]:
# Confusion matrices + macro precision/recall/F1 for the three binary JSON prompts.
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns

JSON_PROMPT_LABELS = {
    'elevator_state':  ['open', 'closed'],
    'stair_direction': ['up', 'down'],
    'restroom_gender': ['male', 'female'],
}

confusion_dump = {}
binary_metrics = {}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (pid, labels) in zip(axes, JSON_PROMPT_LABELS.items()):
    sub = scored_df[(scored_df['prompt_id'] == pid) & scored_df['format_valid']]
    if len(sub) == 0:
        ax.set_title(f'{pid} (no valid outputs)')
        ax.axis('off')
        continue

    cm = confusion_matrix(sub['expected'], sub['extracted'], labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax, cbar=False)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(pid)
    confusion_dump[pid] = {
        'labels': labels,
        'matrix': cm.tolist(),
    }

    p, r, f1, _ = precision_recall_fscore_support(
        sub['expected'], sub['extracted'],
        labels=labels, average='macro', zero_division=0,
    )
    binary_metrics[pid] = {
        'macro_precision': round(float(p),  4),
        'macro_recall':    round(float(r),  4),
        'macro_f1':        round(float(f1), 4),
    }

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

with (RESULTS_DIR / 'confusion_matrices.json').open('w', encoding='utf-8') as f:
    json.dump(confusion_dump, f, indent=2)
with (RESULTS_DIR / 'binary_metrics.json').open('w', encoding='utf-8') as f:
    json.dump(binary_metrics, f, indent=2)

print('Macro precision / recall / F1 for the binary JSON prompts:')
for pid, m in binary_metrics.items():
    print(f'  {pid:18s} P={m["macro_precision"]:.4f}  R={m["macro_recall"]:.4f}  F1={m["macro_f1"]:.4f}')
print(f'\nSaved confusion matrices to {RESULTS_DIR / "confusion_matrices.png"}')
print(f'Saved confusion matrices JSON to {RESULTS_DIR / "confusion_matrices.json"}')
print(f'Saved binary metrics to {RESULTS_DIR / "binary_metrics.json"}')


## 6. Reproducibility log

Single JSON capturing model id, dataset size, GPU, software versions, seed,
decoding params, per-prompt scores, latency stats and confusion matrices.
This is the artefact you cite in the TFG memory and what makes the run
reproducible months from now.


In [ ]:
import platform
import datetime
import sys
import transformers
import PIL

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capability = '.'.join(str(x) for x in torch.cuda.get_device_capability(0))
    gpu_total_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    cuda_version = torch.version.cuda
    cudnn_version = torch.backends.cudnn.version()
else:
    gpu_name = 'CPU'
    gpu_capability = None
    gpu_total_vram_gb = 0.0
    cuda_version = None
    cudnn_version = None

reproducibility = {
    'run_timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'model': {
        'id': MODEL_ID,
        'n_params_billions': N_PARAMS_BILLIONS,
        'dtype': MODEL_DTYPE,
        'load_time_s': MODEL_LOAD_TIME_S,
    },
    'hardware': {
        'device': DEVICE,
        'gpu_name': gpu_name,
        'gpu_capability': gpu_capability,
        'gpu_total_vram_gb': gpu_total_vram_gb,
        'vram_peak_gb_during_run': VRAM_PEAK_GB,
        'cuda_version': cuda_version,
        'cudnn_version': cudnn_version,
    },
    'software': {
        'python': platform.python_version(),
        'platform': platform.platform(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'pillow': PIL.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__,
    },
    'seed': SEED,
    'decoding': {
        'do_sample': False,
        'num_beams': 1,
        'enable_thinking': False,
        'max_new_tokens': MAX_NEW_TOKENS,
    },
    'system_prompt': SYSTEM_PROMPT,
    'dataset': {
        'n_images_evaluated': len(scored_df),
        'per_category': dict(Counter(r['category'] for r in raw_records)),
        'per_prompt':   dict(Counter(r['prompt_id'] for r in raw_records)),
    },
    'total_inference_time_s': TOTAL_INFERENCE_TIME_S,
    'per_prompt_summary': summary_df.to_dict(orient='records'),
    'binary_metrics': binary_metrics,
    'confusion_matrices': confusion_dump,
}

with (RESULTS_DIR / 'reproducibility.json').open('w', encoding='utf-8') as f:
    json.dump(reproducibility, f, indent=2)

print('Reproducibility log written. Snapshot:')
print(f'  Model:    {MODEL_ID} ({N_PARAMS_BILLIONS}B, {MODEL_DTYPE})')
print(f'  GPU:      {gpu_name} ({gpu_total_vram_gb} GB, peak used: {VRAM_PEAK_GB} GB)')
print(f'  Seed:     {SEED}')
print(f'  Software: torch {torch.__version__} | transformers {transformers.__version__} | python {platform.python_version()}')
print(f'  Run UTC:  {reproducibility["run_timestamp_utc"]}')


## 7. Final verdict + download results bundle

Prints the per-prompt summary, applies the validation pass threshold (0.90
on both `format_validity` and `accuracy`), zips `/content/results/` and
triggers a browser download so you don't have to fish files out of the
runtime by hand.


In [ ]:
print('=' * 70)
print(f'EXTRACTION VALIDATION — {MODEL_ID}')
print('=' * 70)
print(summary_df.to_string(index=False))
print()
print(f'Model load:        {MODEL_LOAD_TIME_S} s')
print(f'Peak VRAM:         {VRAM_PEAK_GB} GB ({gpu_name})')
print(f'Total inference:   {TOTAL_INFERENCE_TIME_S} s ({len(scored_df)} images)')
print()
print(f'Artefacts written to {RESULTS_DIR}:')
for p in sorted(RESULTS_DIR.iterdir()):
    print(f'  - {p.name}')
print('=' * 70)

THRESHOLD = 0.90
per_prompt = summary_df[summary_df['prompt_id'] != 'TOTAL']
weak = per_prompt[(per_prompt['format_validity'] < THRESHOLD) | (per_prompt['accuracy'] < THRESHOLD)]
if weak.empty:
    print(f'\nVERDICT: PASS — every prompt >= {THRESHOLD:.0%} on both format_validity and accuracy.')
else:
    print(f'\nVERDICT: REVIEW — the following prompts are below {THRESHOLD:.0%}:')
    print(weak[['prompt_id', 'format_validity', 'accuracy']].to_string(index=False))

# Bundle results into a single zip and trigger browser download.
bundle_path = BASE_DIR / 'extraction_validation_results.zip'
with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS_DIR.rglob('*')):
        if p.is_file():
            z.write(p, arcname=f'results/{p.relative_to(RESULTS_DIR).as_posix()}')

print(f'\nBundled results -> {bundle_path} ({bundle_path.stat().st_size / 1024:.1f} KB)')

try:
    from google.colab import files  # type: ignore
    files.download(str(bundle_path))
    print('Browser download triggered.')
except Exception as e:
    print(f'(Not in Colab or download failed: {e}). Find the bundle at {bundle_path}.')
